In [1]:
# from crewai import agents

In [2]:
from xact.data.chunk import Chunker



Feb/06 22:03:31 |   xact.log.config     | INFO     | log initilized
Feb/06 22:03:31 |   xact.log.config     | XACT_STREAM | log streaming initilized
Feb/06 22:03:31 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json
Feb/06 22:03:31 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json


In [ ]:
data=(
    """

MongoDB Developer Center
Developer Topics
Products
Atlas
Tutorials
How to Choose the Right Chunking Strategy for Your LLM Application

Choosing the right chunking strategy for your RAG application
Chunking strategies
Step 1: Install required libraries
Step 2: Set up pre-requisites
Step 3: Load the dataset
Step 4: Define chunking functions
Step 5: Generate the evaluation dataset
Step 6: Evaluate chunking strategies
Conclusion
MongoDB logo
Language Selector Icon

English
© 2024 MongoDB, Inc.
About

Careers
Investor Relations
Legal
GitHub
Security Information
Trust Center
Connect with Us
Support

Contact Us
Customer Portal
Atlas Status
Customer Support
Manage Cookies

"""
)

In [4]:
from xact.data.data import DataX


ch = Chunker()

ch.chunk_document(DataX(content=data))[0].model_dump_json()

'{"uid":"c143d7c9-5e07-4489-a9f2-aa5422acd3ec","cid":"c143d7c9-5e07-4489-a9f2-aa5422acd3ec_1","flow_mode":"prompt","role":"xact","content":" MongoDB Developer Center Developer Topics Products Atlas Tutorials How to Choose the Right Chunking Strategy for Your LLM Application Apoorva Joshi 16 min read • Published Jun 18, 2024 • Updated Jun 18, 2024 AI Python Atlas Copy link Facebook Icon twitter icon linkedin icon Rate this tutorial star-empty star-empty star-empty star-empty star-empty In Part 1 of this series on Retrieval Augmented Generation (RAG), we looked into choosing the right embedding model for your RAG application. While the choice of embedding model is an important consideration to ensure good quality retrieval for RAG, there is one key decision to be made before the embedding stage that can have a significant downstream impact — choosing the right chunking strategy for your data. In this tutorial, we will cover the following: What is chunking and why is it important for RAG?

In [5]:


from xact.log.config import log_manager


log = log_manager.init(__name__)

from xact.llm.tool import gen_function_schema

class Tool:
    def __init__(
        self,
        func: callable = None,
        description: str = None,
        param_description: dict = None,
    ):
        if not callable(func):
            raise ValueError("The 'func' argument must be a callable function.")
        
        self.func = func
        self.description = description
        self.param_description = param_description or {}
        self.fun_schema = gen_function_schema(func)

        # Dynamically set the docstring and annotations
        self.__doc__ = func.__doc__
        self.__annotations__ = func.__annotations__

    def run(self, *args, **kwargs):
        """Execute the function and log its execution."""
        name = self.fun_schema["function"]["name"]
        log.info(f"Executing tool: {name}")
        return self.func(*args, **kwargs)

    def __call__(self, *args, **kwargs):
        """Allow the Tool instance to be called like a function."""
        return self.run(*args, **kwargs)

    def get_schema(self, is_param_description: bool = True):
        """Generate and return the schema for the function."""
        fun_schema = self.fun_schema.copy()

        if is_param_description:
            if self.param_description:
                for k, v in fun_schema["function"]["parameters"]["properties"].items():
                    if k in self.param_description:
                        fun_schema["function"]["parameters"]["properties"][k][
                            "description"
                        ] = self.param_description[k]

        if self.description:
            fun_schema["function"]["description"] = (
                fun_schema["function"]["description"] + " \n " + self.description
            )

        return fun_schema

In [7]:
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# Create a Tool instance
tool = Tool(
    func=add,
)

# Access the docstring and annotations
print(tool.__doc__)  # Output: Add two numbers.
print(tool.__annotations__)  # Output: {'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}

# Execute the function
result = tool(3, 5)

print(result)  # Output: 8

# Get the schema
schema = tool.get_schema()
print(schema)

Feb/06 22:11:44 |       __main__        | INFO     | Executing tool: add


Add two numbers.
{'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}
8
{'type': 'function', 'function': {'name': 'add', 'description': 'Add two numbers.', 'parameters': {'type': 'object', 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}}}
